# 00 · Dataset overview

What the EXIST 2026 corpus looks like before any modelling: how many instances
carry a usable hard label, how the annotators disagree, and how the five sexism
categories are distributed across the two languages.

Two facts drive most design decisions downstream and are worth seeing directly:

1. **The vote thresholds differ between modalities.** Memes have six annotators
   and need `>3` / `>2` / `>1` votes; videos have three and need `>1`
   throughout. A sizeable share of instances therefore has *no* hard label at
   all, and those instances are undecided rather than negative.
2. **Disagreement is the signal, not noise.** Under *Learning with
   Disagreements* the target is the annotator distribution, so the histogram of
   agreement below is what the soft metric actually scores.

Set `MODALITY` and re-run to switch between memes and videos.

In [ ]:
# Makes the notebook work from a clone (no install) and on Colab alike.
import sys
from pathlib import Path

REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO / "src"))

# Point this at the directory holding the EXIST 2026 corpus. The configs read
# it from here, so nothing below contains a hard-coded path.
import os
os.environ.setdefault("EXIST2026_ROOT", str(Path.home() / "EXIST_2026"))

import pandas as pd
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 160)

In [ ]:
MODALITY = "memes"  # "memes" | "videos"

from exist2026.config import load_fewshot_config
from exist2026.datasets import describe, load_corpus

config = load_fewshot_config(f"fewshot_{MODALITY}", stage="sanity")
corpus = load_corpus(
    config.modality,
    config.thresholds,
    train_json=config.data.train_json(),
    train_media_dir=config.data.train_media_dir(),
    require_media=False,   # an overview does not need the media on disk
)
print(f"{config.modality.value}: {len(corpus.frame):,} training instances")
print(f"official thresholds: {config.thresholds}")
describe(corpus)

## Hard labels: how much of the corpus is decided

An instance below the threshold has no hard label. It still contributes to the
*soft* targets — it is simply not scorable in the hard-hard setting.

In [ ]:
import pandas as pd
from exist2026.taxonomy import SEXISM_CATEGORIES, Subtask

rows = []
for subtask in (Subtask.IDENTIFICATION, Subtask.INTENTION):
    column = corpus.hard_column(subtask)
    decided = corpus.train[column].notna().sum()
    rows.append({
        "subtask": column.removeprefix("hard_"),
        "decided": int(decided),
        "undecided": int(len(corpus.train) - decided),
        "pct_decided": round(100 * decided / len(corpus.train), 1),
    })
pd.DataFrame(rows)

In [ ]:
# Class balance of x.1, per language.
identification = corpus.hard_column(Subtask.IDENTIFICATION)
decided = corpus.train[corpus.train[identification].notna()].copy()
decided["label"] = decided[identification].map({0: "NO", 1: "YES"})
pd.crosstab(decided["lang"], decided["label"], margins=True)

## Category distribution

Categories co-occur: roughly half of the sexist instances carry more than one.
That is why x.3 is trained with BCE over independent probabilities rather than
a softmax, and why the few-shot pool reserves exemplars for genuine
co-occurrence.

In [ ]:
from exist2026.labels.hard import active_categories

category_column = corpus.hard_column(Subtask.CATEGORIZATION)
active = [active_categories(vector) for vector in corpus.train[category_column]]

counts = pd.Series(
    [c for categories in active for c in categories], dtype="object"
).value_counts().reindex(SEXISM_CATEGORIES).fillna(0).astype(int)

cardinality = pd.Series([len(c) for c in active]).value_counts().sort_index()
cardinality.index.name = "categories per instance"

display(counts.to_frame("instances"))
display(cardinality.to_frame("instances"))

## Annotator agreement

The share of annotators backing the majority option. A corpus concentrated at
1.0 would make the soft and hard settings nearly equivalent; this one is not.

In [ ]:
import matplotlib.pyplot as plt

from exist2026.labels.hard import identification_consensus

agreement = [
    identification_consensus(corpus.annotations(instance_id, Subtask.IDENTIFICATION))
    for instance_id in corpus.train["id_EXIST"]
]

figure, axis = plt.subplots(figsize=(7, 3.5))
axis.hist(agreement, bins=12, color="#2C5F8A", edgecolor="white")
axis.set_xlabel("share of annotators agreeing with the majority (x.1)")
axis.set_ylabel("instances")
axis.set_title(f"Annotator agreement - {config.modality.value}")
axis.grid(alpha=0.3)
figure.tight_layout()

## Text length

The enriched `capsanocr` view is much longer than the raw on-screen text, which
is why the trained encoders use a 256 (memes) / 512 (videos) token window. Here
we only look at the released text, as a lower bound.

In [ ]:
lengths = corpus.train["text"].str.split().str.len().fillna(0)
print(lengths.describe().round(1).to_string())
print(f"\ninstances with no released text: {(lengths == 0).sum():,}")